## 1. Environment Setup
Clone the AquaMind repo from GitHub and install all required dependencies.

In [ ]:
# ── Cell 0 — Config ───────────────────────────────────────────────────────────
# Central config for the entire notebook.
# Change values here — all cells below will use them automatically.

REPO_PATH    = '/kaggle/working/AquaMind'                    # where the repo will be cloned
EPOCHS       = 50                                             # number of training epochs
IMGSZ        = 640                                            # input image size for YOLOv8
RUN_NAME     = 'run_data_ann_5r_8c_9r_10r_11c_14c_17g_2026_08_04_2006' # name of this training run

print(f"Config loaded: REPO_PATH={REPO_PATH}, EPOCHS={EPOCHS}, IMGSZ={IMGSZ}, RUN_NAME={RUN_NAME}")


In [ ]:
# ── Cell 1 — git clone ───────────────────────────────────────────────────────────

%cd /kaggle/working
!rm -rf AquaMind
!git clone https://github.com/govinda-lienart/AquaMind.git
%cd /kaggle/working/AquaMind
print()
print("GitHub repo cloned successfully.")

In [ ]:
# ── Cell 2 — Install dependencies ────────────────────────────────────────────
# ultralytics  → YOLOv8 training and inference
# dvc          → data version control (tracks dataset versions like git tracks code)
# dvc-gdrive   → DVC plugin to use Google Drive as the remote storage backend

!pip install ultralytics dvc dvc-gdrive -q

import ultralytics, dvc
print(f"ultralytics: {ultralytics.__version__}")
print(f"dvc:         {dvc.__version__}\n")
print("All dependencies installed.")

## 2. Data Pipeline
Load the dataset from Google Drive via DVC, then verify the structure is correct before training.

In [ ]:
# ── Cell 3 — Inspect dataset.yaml ────────────────────────────────────────────
# dataset.yaml tells YOLOv8 where to find the data and what classes exist.
# path   → root folder of the dataset (relative to this repo)
# train  → subfolder containing training images
# val    → subfolder containing validation images
# nc     → number of classes
# names  → class labels (index 0 = danio_rerio, index 1 = reflection)

with open('/kaggle/working/AquaMind/dataset.yaml', 'r') as f:
    print(f.read())

In [ ]:
# ── Cell 4 — Load service account google drive + save credentials ─────────────────────────
from kaggle_secrets import UserSecretsClient
import os

secret = UserSecretsClient().get_secret("GDRIVE_SA_KEY")
os.makedirs('/kaggle/working/AquaMind/secrets', exist_ok=True)

creds_path = '/kaggle/working/AquaMind/secrets/aquamind-497014-5b34149af3fb.json'
with open(creds_path, 'w') as f:
    f.write(secret)

if os.path.exists(creds_path) and os.path.getsize(creds_path) > 0:
    print(f"Service account saved. ({os.path.getsize(creds_path)} bytes)")
else:
    print("ERROR: credentials file missing or empty!")

In [ ]:
# ── Cell 5 — Inspect dataset.dvc ────────────────────────────────────────────
# dataset.dvc is the DVC pointer file committed to git.
# It contains the md5 hash of the dataset — DVC uses this hash to fetch
# the correct version of the data from Google Drive.
# If this hash is old, DVC will pull the wrong dataset.

with open('/kaggle/working/AquaMind/dataset.dvc', 'r') as f:
    print(f.read())

In [ ]:
# ── Cell 6 — DVC pull ───────────────────────────────────────────────────────
# DVC reads dataset.dvc, finds the md5 hash, and downloads the matching
# dataset version from Google Drive into the dataset/ folder.
# -q flag suppresses verbose output — prints only when complete.

!dvc pull -q
print("DVC pull complete.")

In [ ]:
# ── Cell 7 — Verify dataset ───────────────────────────────────────────────────
import yaml, os

with open(f'{REPO_PATH}/dataset.yaml') as f:
    cfg = yaml.safe_load(f)

base = f"{REPO_PATH}/{cfg['path']}"
print(f"Dataset: {cfg['path']}")
print(f"Train:   {len(os.listdir(base + '/images/train'))} images")
print(f"Val:     {len(os.listdir(base + '/images/val'))} images")


## 3. Training
Fine-tune YOLOv8s on the AquaMind zebrafish dataset.

In [ ]:
# ── Cell 8 — Train YOLOv8 ────────────────────────────────────────────────────
# yolov8s.pt  → pretrained small YOLOv8 model (good balance speed/accuracy)
# data        → path to dataset.yaml (class names + train/val paths)
# epochs      → number of full passes through the training data
# imgsz       → all images resized to 640x640 before being fed to the model
# project     → output folder for results (weights, metrics, plots)
# name        → subfolder name for this specific run

!yolo detect train \
  data={REPO_PATH}/dataset.yaml \
  model=yolov8s.pt \
  epochs={EPOCHS} \
  imgsz={IMGSZ} \
  project=/kaggle/working/aquamind_yolov8 \
  name={RUN_NAME}

import os
weights_path = f'/kaggle/working/aquamind_yolov8/{RUN_NAME}/weights/best.pt'
if os.path.exists(weights_path):
    print(f'Training complete. Best weights saved to {weights_path}')
else:
    print('WARNING: Training may have failed — best.pt not found.')

## 4. Export Results
Zip the training output and download it from the Output panel.

In [ ]:
# ── Cell 9 — Zip results ──────────────────────────────────────────────────────
# Packages the full run output folder into a zip file.
# Contains: best.pt (best weights), last.pt, metrics, confusion matrix, plots.
# Download it from the Kaggle Output panel on the right.

import shutil
from datetime import datetime

date_str = datetime.now().strftime('%Y%m%d_%H%M')
run_dir  = f'/kaggle/working/aquamind_yolov8/{RUN_NAME}'
zip_name = f'/kaggle/working/aquamind_yolov8_{RUN_NAME}_{date_str}'

shutil.make_archive(zip_name, 'zip', run_dir)
print(f'Zip ready: {zip_name}.zip')
print('Download it from the Output panel on the right.')